# Import Modules

In [ ]:
#high level modules
import os
import importlib.util
from functools import partial
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:

def import_from_path(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

fun_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/functions/"
this_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/forecast_rollout/"

import_from_path("universals", os.path.join(fun_dir, "universal_functions.py"))
from universals import load_pickle_file

import_from_path("forecast_fx", os.path.join(this_dir, "forecast_functions.py"))
from forecast_fx import prep_features, static_regime, pulsed_regime, rollout_forecast

dump_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/tiny_model/test_forecast/"


# Import models

In [ ]:
model_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/tiny_model/dump/five_ten/"

model_1 = load_pickle_file("model_1.pkl", model_dir)
model_2 = load_pickle_file("model_2.pkl", model_dir)
model_3 = load_pickle_file("model_3.pkl", model_dir)
model_4 = load_pickle_file("model_4.pkl", model_dir)

# Import data

In [ ]:
data_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/data/"

test = pd.read_csv(os.path.join(data_dir, "test.csv"))

# reduce to fewer variables
drop_cols = ['total_solar_radiation_m2', 'mean_rel_hum_m2', 
       'pump_cfs_m2', 'pump_cfs_m3', 'nf_cfs_m2', 'nf_cfs_m3', 'nf_cfs_m4',
       'chipmunk_cfs_m2', 'chipmunk_cfs_m3', 'chipmunk_cfs_m4']

test = test.drop(columns = drop_cols)

test["date"] = pd.to_datetime(test["date"])

# filter test set for data from 2015, then from 2022
test_2015 = test[test["date"].dt.year == 2015].copy()
test_2022 = test[test["date"].dt.year == 2022].copy()

print(test_2022)

# Roll out forecast

Here, we're just iteratively modeling, for each of the 4 models for the next 7 days. First, we need to remove any columns that aren't in the model proper, and we need to make sure the order of the columns is the same as the training set. To do this, we'll just select the feature names in the proper order.

First thing is that we need to prep the data for forecasting. This step uses a single date and grabs the data seven days in the future. Data are recoded as needed, and a forecast date column is created - this is the date that the forecast is made. 

In [ ]:

all_dates_2015 = pd.date_range(start="2015-07-01", end="2015-10-08", freq="D").strftime("%Y-%m-%d")

control_dataset = pd.concat([prep_features(one_date=d, data=test_2015, regime="control") for d in all_dates_2015])
control_dataset["forecast_date"] = pd.to_datetime(control_dataset["forecast_date"])


In [ ]:
# get the feature names from the operational training file
upstream_names = pd.read_csv(os.path.join(data_dir, "training_1.csv"))
# reduce to fewer variables
drop_cols = ['total_solar_radiation_m2', 'mean_rel_hum_m2', 
       'pump_cfs_m2', 'pump_cfs_m3', 'nf_cfs_m2', 'nf_cfs_m3', 'nf_cfs_m4',
       'chipmunk_cfs_m2', 'chipmunk_cfs_m3', 'chipmunk_cfs_m4']

feature_names = upstream_names.drop(columns = drop_cols).columns

# remove date, and two target columns (first 3 columns)
feature_names = feature_names[3:]
feature_names = list(feature_names) + ["forecast_date"]

# and get those features in order for each dataset! (but we also need the forecast date!)
control_for_model = control_dataset[feature_names].copy()

unique_forecast = control_for_model["forecast_date"].unique()

And now rollout the forecast per unique valid date.

In [ ]:
control_forecasts = (pd.concat([rollout_forecast(data = control_for_model, 
                                                 m1 = model_1, m2 = model_2, m3 = model_3, m4 = model_4, 
                                                 fore_date = d) for d in unique_forecast]))

And now we need to transform the data back to temp. To do this we need the transformation file.

In [ ]:
transformations = pd.read_csv("/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/data/mu_sigma.csv", index_col=0)


In [ ]:
# we need to back transform the data for mean 1m and mean 0.5m
control_forecasts["mean_1m_temp_degC"] = control_forecasts["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
control_forecasts["mean_0_5m_temp_degC"] = control_forecasts["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

export_features = ["forecast_date", "valid_date", "model", "mean_1m_temp_degC", "mean_0_5m_temp_degC"]
control_forecasts[export_features]
# save the files to csv
control_forecasts.to_csv(os.path.join(dump_dir, "forecasted_temp_2015_control_collated.csv"), index=False)

In [ ]:
# Ensure valid_date is datetime
control_forecasts["valid_date"] = pd.to_datetime(control_forecasts["valid_date"])
control_forecasts["forecast_date"] = pd.to_datetime(control_forecasts["forecast_date"])

observations = test_2015.copy()
observations["date"] = pd.to_datetime(observations["date"])
observations["mean_1m_temp_degC"] = observations["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]

# Group by forecast_date and valid_date, then aggregate
summary = control_forecasts.groupby(["forecast_date", "valid_date"]).agg(
    mean_temp=("mean_1m_temp_degC", "mean"),
    min_temp=("mean_1m_temp_degC", "min"),
    max_temp=("mean_1m_temp_degC", "max")
).reset_index()

# Plot
plt.figure(figsize=(12, 6))

# Loop over forecast_date to plot each forecast run
for forecast_date, group in summary.groupby("forecast_date"):
    plt.plot(group["valid_date"], group["mean_temp"], label=f"{forecast_date.date()}")
    plt.fill_between(group["valid_date"], group["min_temp"], group["max_temp"], alpha=0.2)

plt.scatter(observations["date"], observations["mean_1m_temp_degC"], color='black', label='Observed', zorder=5)

plt.xlabel("Valid Date")
plt.ylabel("Mean 1m Temp (°C)")
plt.title("Forecast Evolution by Initialization Date")
# plt.legend(title="Forecast Date", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

all_dates_2022 = pd.date_range(start="2022-05-27", end="2022-10-02", freq="D").strftime("%Y-%m-%d")

control_dataset = pd.concat([prep_features(one_date=d, data=test_2022, regime="control") for d in all_dates_2022])
control_dataset["forecast_date"] = pd.to_datetime(control_dataset["forecast_date"])


In [ ]:
# and get those features in order for each dataset! (but we also need the forecast date!)
control_for_model = control_dataset[feature_names].copy()

unique_forecast = control_for_model["forecast_date"].unique()

In [ ]:
control_forecasts = (pd.concat([rollout_forecast(data = control_for_model, 
                                                 m1 = model_1, m2 = model_2, m3 = model_3, m4 = model_4, 
                                                 fore_date = d) for d in unique_forecast]))

In [ ]:
# we need to back transform the data for mean 1m and mean 0.5m
control_forecasts["mean_1m_temp_degC"] = control_forecasts["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
control_forecasts["mean_0_5m_temp_degC"] = control_forecasts["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

export_features = ["forecast_date", "valid_date", "model", "mean_1m_temp_degC", "mean_0_5m_temp_degC"]
control_forecasts[export_features]
# save the files to csv
control_forecasts.to_csv(os.path.join(dump_dir, "forecasted_temp_2022_control_collated.csv"), index=False)

In [ ]:
# Ensure valid_date is datetime
control_forecasts["valid_date"] = pd.to_datetime(control_forecasts["valid_date"])
control_forecasts["forecast_date"] = pd.to_datetime(control_forecasts["forecast_date"])

observations = test_2022.copy()
observations["date"] = pd.to_datetime(test_2022["date"])
observations["mean_1m_temp_degC"] = observations["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]

# Group by forecast_date and valid_date, then aggregate
summary = control_forecasts.groupby(["forecast_date", "valid_date"]).agg(
    mean_temp=("mean_1m_temp_degC", "mean"),
    min_temp=("mean_1m_temp_degC", "min"),
    max_temp=("mean_1m_temp_degC", "max")
).reset_index()

# Plot
plt.figure(figsize=(12, 6))

# Loop over forecast_date to plot each forecast run
for forecast_date, group in summary.groupby("forecast_date"):
    plt.plot(group["valid_date"], group["mean_temp"], label=f"{forecast_date.date()}")
    plt.fill_between(group["valid_date"], group["min_temp"], group["max_temp"], alpha=0.2)

plt.scatter(observations["date"], observations["mean_1m_temp_degC"], color='black', label='Observed', zorder=5)

plt.xlabel("Valid Date")
plt.ylabel("Mean 1m Temp (°C)")
plt.title("Forecast Evolution by Initialization Date")
# plt.legend(title="Forecast Date", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()